# Experimento pareado: selección por F1 frente a F2 en LSTM32 dropout 0,2 — Kaggle

Copia para Kaggle del experimento F1/F2. Se comparan dos entrenamientos desde cero con los mismos datos, arquitectura, semilla, BCE, Adam, learning rate y regularización. Solo cambia el objetivo de selección: F2 frente a F1 para elegir checkpoint y umbral.

El entrenamiento exige GPU CUDA. 2025 permanece reservado. La versión Colab original se conserva en `08_lstm_f1_colab.ipynb`.


## 1. Importar notebook y adjuntar el paquete

Importa este archivo en Kaggle con **File → Import Notebook**. En **Settings → Accelerator**, selecciona **GPU**. En el panel **Input → Add Input → Upload**, adjunta `runs/lstm_colab_f1_v2.zip`. Ejecuta esta sección para verificar el paquete; la corrida rechazará un ZIP con hash distinto.


In [ ]:
from pathlib import Path
import hashlib, json, stat, zipfile

INPUT_ROOT = Path('/kaggle/input')
ROOT = Path('/kaggle/working/f1-lstm-f1-v2')
EXPECTED_PACKAGE_SHA256 = '0023a176c7f3298b2df5a0c731495c6dfc073513ef51b2e2dd26eb71e8956a3c'
if ROOT.exists(): raise FileExistsError('Usa una sesión/salida limpia: /kaggle/working/f1-lstm-f1-v2 ya existe.')
packages = sorted(INPUT_ROOT.rglob('lstm_colab_f1_v2.zip'))
if len(packages) != 1:
    raise FileNotFoundError('Adjunta exactamente lstm_colab_f1_v2.zip mediante Input → Add Input → Upload.')
PACKAGE = packages[0]
def sha256(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''): h.update(chunk)
    return h.hexdigest()
package_hash = sha256(PACKAGE)
print('Paquete encontrado:', PACKAGE)
print('SHA-256 del paquete:', package_hash)
if package_hash != EXPECTED_PACKAGE_SHA256: raise ValueError('El hash no coincide con el ZIP v2 corregido.')
with zipfile.ZipFile(PACKAGE) as z:
    members=z.infolist(); names=set()
    for m in members:
        p=Path(m.filename)
        if p.is_absolute() or '..' in p.parts or chr(92) in m.filename or stat.S_ISLNK(m.external_attr >> 16): raise ValueError(f'Ruta ZIP no segura: {m.filename}')
        if m.filename in names: raise ValueError(f'Miembro duplicado: {m.filename}')
        names.add(m.filename)
    manifest=json.loads(z.read('colab_package_manifest.json'))
    if manifest.get('artifact_version') != 'lstm_colab_f1_inputs_v1' or manifest.get('test_arrays_included') is not False: raise ValueError('Contrato de paquete no admitido o contiene test.')
    expected=set(manifest['files_sha256']) | {'colab_package_manifest.json'}
    if names != expected: raise ValueError('Los archivos del ZIP no coinciden con su manifiesto.')
    ROOT.mkdir(parents=True)
    z.extractall(ROOT)
for name, digest in manifest['files_sha256'].items():
    if sha256(ROOT/name) != digest: raise ValueError(f'SHA-256 incorrecto: {name}')
print('Manifiesto verificado:', len(manifest['files_sha256']), 'archivos; sin datos de prueba.')


## 2. Instalar dependencias y confirmar GPU

La ejecución se detiene antes de entrenar si Kaggle no expone CUDA. En Settings debe estar seleccionado GPU; no se permite fallback a CPU. PyTorch lo proporciona el entorno acelerado de Kaggle y no se reinstala.


In [ ]:
import subprocess, sys
from pathlib import Path
ROOT = Path('/kaggle/working/f1-lstm-f1-v2')
if not (ROOT / 'requirements-colab-experiments.txt').is_file():
    raise RuntimeError('No encuentro el paquete extraído en /kaggle/working/f1-lstm-f1-v2. Ejecuta primero la sección 1.')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT/'requirements-colab-experiments.txt')], check=True)
import torch
cuda_available = torch.cuda.is_available()
print('PyTorch:', torch.__version__, '| build CUDA:', torch.version.cuda, '| cuda.is_available:', cuda_available, '| dispositivos visibles:', torch.cuda.device_count())
if not cuda_available:
    if torch.version.cuda is None:
        detail = 'Este entorno de Kaggle tiene un build CPU de PyTorch.'
    else:
        detail = 'PyTorch tiene soporte CUDA, pero Kaggle no conectó una GPU a esta sesión.'
    raise RuntimeError(detail + ' Selecciona GPU en Settings → Accelerator y vuelve a iniciar la sesión; no se instalará ni sustituirá torch.')
print('GPU:', torch.cuda.get_device_name(0))


## 3. Entrenar la pareja

La referencia selecciona checkpoint y umbral por F2; la candidata lo hace por F1. Ambas usan early stopping en validación a umbral fijo 0,5, paciencia seis y desempate por BCE menor. El runner valida que solo cambie el objetivo de selección y comprueba la misma GPU/entorno entre las dos corridas. Si falla, la celda imprime `status.json`. La validación es de desarrollo, no prueba independiente.


In [ ]:
import json, subprocess, sys
from pathlib import Path
from IPython.display import FileLink, display
ROOT = Path('/kaggle/working/f1-lstm-f1-v2')
if not (ROOT / 'scripts/entrenar_comparacion_lstm_f1.py').is_file():
    raise RuntimeError('No encuentro el runner corregido del paquete v2. Ejecuta primero la sección 1.')
RUN = ROOT/'runs/lstm32_f1_objective_v1_seed42'
if RUN.exists() or RUN.with_suffix('.zip').exists(): raise FileExistsError('La salida ya existe. Conserva los artefactos y usa una sesión/salida nueva.')
result = subprocess.run(
    [sys.executable, 'scripts/entrenar_comparacion_lstm_f1.py', '--reference-config', 'configs/experimento_lstm32_f1_ref_v1.json', '--candidate-config', 'configs/experimento_lstm32_f1_candidate_v1.json', '--output', str(RUN.relative_to(ROOT))],
    cwd=ROOT,
    check=False,
)
if result.returncode != 0:
    status_path = RUN/'status.json'
    if status_path.is_file():
        print('Estado del par:')
        print(json.dumps(json.loads(status_path.read_text()), indent=2))
    raise RuntimeError(f'La corrida falló con código {result.returncode}; revisa el estado anterior antes de reintentar.')
summary=json.loads((RUN/'pair_summary.json').read_text())
print(json.dumps(summary, indent=2))
print('ZIP de resultados:', RUN.with_suffix('.zip'))
display(FileLink(str(RUN.with_suffix('.zip'))))


## 4. Devolver resultados al proyecto

Descarga el ZIP de resultados desde el enlace que aparece al final de la sección 3 o desde el panel **Output** de Kaggle. Guárdalo intacto como `runs/lstm32_f1_objective_v1_seed42.zip` en el repositorio local para verificar hashes y paridad en CPU. El test 2025 no se utilizó.
